## Registering LGMS COGs as Earth Engine Assets

Created by Mel Rose; Last updated 11/25/2026

## Setup

In [1]:
#TODO: This script is behind the current model version (1.0.6) Change to cn paths after merging to updated model branch.
#TODO: Add in other datasets (land state node, land use, organic soil state, combined LULUCF, etc)
#TODO: Update with cn patterns

from google.cloud import storage
import ee
import json
from pprint import pprint
import google.auth
from google.auth.transport.requests import AuthorizedSession
from src.utilities import constants_and_names as cn

# --------------------------------------------------------------------------------------------------
# USER SELECTIONS
# Leave as None to run all available options.

# Options: ["vegetation", "mineral_soil", "organic_soil"]
datasets = ["vegetation", "organic_soil"]

# Options: ["emissions", "removals", "netflux"]
fluxes = ["emissions"]

# Options: ["all_gases", "CO2_only", "non_CO2_only"]
# If gases are None it defaults to all_gases.
# Removals and mineral soil are CO2 only, so they are included for all_gases or CO2_only.
gases = ["CO2_only"]

# Options: [2016, 2017, 2018, 2019, 2020]
# If years are None it creates a stacked band image of all years for a dataset
# Soil years are resolved below so that if you run [2016, 2017, 2018] it will use the 2016_2020 interval and create just a single EE asset registration for that interval\
years = None

print(f"Datasets selected: {datasets}")
print(f"Fluxes selected:   {fluxes}")
print(f"Gases selected:    {gases}")
print(f"Years selected:    {years}")
# --------------------------------------------------------------------------------------------------

Datasets selected: ['vegetation', 'organic_soil']
Fluxes selected:   ['emissions']
Gases selected:    ['CO2_only']
Years selected:    None


## Constants

In [2]:
veg_model_version_underscore = "1_0_5"
ee_project = "landandcarbon"
gcs_bucket = "wri-lcl-lgms"

veg_asset_folder = f"vegetation/v{veg_model_version_underscore}"
min_soil_asset_folder = f"soil/mineral/v{cn.SOC_model_version_underscore}"
org_soil_asset_folder = f"soil/organic/v{cn.organic_soil_model_version_underscore}"

gases = ["all_gases"] if gases is None else gases

all_vegetation_years = list(range(2016, 2025))
stack_vegetation_years = years is None or sorted(years) == all_vegetation_years
years = all_vegetation_years if years is None else sorted(years)

# Resolve soil years
mineral_soil_years = ["2016_2020"]
organic_soil_years = sorted({cn.organic_soil_year_map[year] for year in years})

# Vegetation patterns (same as create_cogs_download_tiles.py)
# Note: the patterns include veg_ at the beginning starting with v1.0.6
veg_emissions_all_gases_pattern =    "gross_emissions__all_C_pools__all_gases__MgCO2e"
veg_emissions_CO2_only_pattern =     "gross_emissions__all_C_pools__CO2_only__MgCO2"
veg_emissions_non_CO2_only_pattern = "gross_emissions__all_C_pools__non_CO2_only__MgCO2e"

veg_netflux_all_gases_pattern =    "net_flux__all_C_pools__all_gases__MgCO2e"
veg_netflux_CO2_only_pattern =     "net_flux__all_C_pools__CO2_only__MgCO2"
veg_netflux_non_CO2_only_pattern = "net_flux__all_C_pools__non_CO2_only__MgCO2e"

veg_removals_pattern = "gross_removals__all_C_pools__MgCO2"  # CO2 only

# Organic soil patterns (same as create_cogs_download_tiles.py)
org_soil_burned_all_gases_pattern = cn.burned_organic_soils_total_pattern
org_soil_burned_CO2_only_pattern = "burned_co2_Mg_CO2_ha_yr"
# org_soil_burned_non_CO2_only_pattern = # TODO: burned organic soils, non-CO2 only

org_soil_drained_all_gases_pattern = cn.drained_organic_soils_total_pattern
org_soil_drained_CO2_only_pattern = "drained_co2_Mg_CO2_ha_yr"
# org_soil_drained_non_CO2_only_pattern = # TODO: drained organic soils, non-CO2 only

In [3]:
credentials, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/cloud-platform'
    ]
)

credentials = credentials.with_quota_project(ee_project)
ee.Initialize(credentials=credentials, project=ee_project)
session = AuthorizedSession(credentials)
storage_client = storage.Client(project=ee_project)
bucket = storage_client.get_bucket(gcs_bucket)

/home/melrose/miniforge3/envs/zonal_stats/lib/python3.13/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/melrose/miniforge3/envs/zonal_stats/lib/python3.13/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Build registration specifications


In [4]:
def interval_bounds(period):
    # Annual vegetation outputs
    if isinstance(period, int):
        return (
            f"{period}-01-01T00:00:00.000000000Z",
            f"{period + 1}-01-01T00:00:00.000000000Z",
        )

    # Multi-year soil outputs such as 2016_2020 or 2021_2024
    start_year, end_year = map(int, str(period).split("_"))
    return (
        f"{start_year}-01-01T00:00:00.000000000Z",
        f"{end_year + 1}-01-01T00:00:00.000000000Z",
    )


asset_specs = {}

# Add a normal one-COG / one-band asset specification.
def add_spec(key, dataset, flux, gas, period, pattern, gcs_dir, ee_folder, version, component=None):
    filename = f"{pattern}_{period}.tif"
    gcs_uri = f"gs://{gcs_bucket}/{gcs_dir.strip('/')}/{filename}"
    asset_name = f"{pattern}_{period}"
    asset_id = f"projects/{ee_project}/assets/{ee_folder.strip('/')}/{asset_name}"
    start_time, end_time = interval_bounds(period)

    asset_specs[key] = {
        "dataset": dataset,
        "flux": flux,
        "gas": gas,
        "component": component,
        "period": period,
        "periods": [period],
        "version": version,
        "filename": filename,
        "gcs_uri": gcs_uri,
        "gcs_uris": [gcs_uri],
        "stacked": False,
        "asset_folder": ee_folder.strip("/"),
        "asset_name": asset_name,
        "asset_id": asset_id,
        "start_time": start_time,
        "end_time": end_time,
    }

# Add one multi-band asset backed by one single-band COG per year.
def add_stacked_spec(
    key,
    dataset,
    flux,
    gas,
    periods,
    pattern,
    gcs_dir,
    ee_folder,
    version,
    component=None
):
    periods = sorted(periods)

    gcs_uris = [
        f"gs://{gcs_bucket}/{gcs_dir.strip('/')}/{pattern}_{period}.tif"
        for period in periods
    ]

    # Build overall period label.
    # Annual years: [2016, ..., 2024] -> 2016_2024
    # Multi-year intervals: ["2016_2020", "2021_2024"] -> 2016_2024
    if isinstance(periods[0], int):
        start_year = periods[0]
        end_year = periods[-1]
    else:
        start_year = int(str(periods[0]).split("_")[0])
        end_year = int(str(periods[-1]).split("_")[-1])

    period_label = f"{start_year}_{end_year}"

    asset_name = f"{pattern}_{period_label}"
    asset_id = (
        f"projects/{ee_project}/assets/"
        f"{ee_folder.strip('/')}/{asset_name}"
    )

    start_time, _ = interval_bounds(periods[0])
    _, end_time = interval_bounds(periods[-1])

    asset_specs[key] = {
        "dataset": dataset,
        "flux": flux,
        "gas": gas,
        "component": component,
        "period": period_label,
        "periods": periods,
        "version": version,
        "filename": None,
        "gcs_uri": None,
        "gcs_uris": gcs_uris,
        "stacked": True,
        "asset_folder": ee_folder.strip("/"),
        "asset_name": asset_name,
        "asset_id": asset_id,
        "start_time": start_time,
        "end_time": end_time,
    }


# --------------------------------------------------------------------------------------------------
# Vegetation
# Each entry: (flux, gas, base pattern, flux folder in GCS and EE)
vegetation_configs = []

if "all_gases" in gases:
    vegetation_configs += [
        ("emissions", "all_gases", veg_emissions_all_gases_pattern, "emissions"),
        ("netflux",   "all_gases", veg_netflux_all_gases_pattern,   "net_flux"),
    ]

if "CO2_only" in gases:
    vegetation_configs += [
        ("emissions", "CO2_only", veg_emissions_CO2_only_pattern, "emissions"),
        ("netflux",   "CO2_only", veg_netflux_CO2_only_pattern,   "net_flux"),
    ]

if "non_CO2_only" in gases:
    vegetation_configs += [
        ("emissions", "non_CO2_only", veg_emissions_non_CO2_only_pattern, "emissions"),
        ("netflux",   "non_CO2_only", veg_netflux_non_CO2_only_pattern,   "net_flux"),
    ]

# Removals are CO2 only, so include them for all_gases or CO2_only (added once)
if "all_gases" in gases or "CO2_only" in gases:
    vegetation_configs += [
        ("removals", "CO2_only", veg_removals_pattern, "removals"),
    ]

if "vegetation" in datasets:
    for flux, gas, base_pattern, flux_folder in vegetation_configs:
        if flux not in fluxes:
            continue

        pattern = f"{base_pattern}_ha_yr"
        gcs_dir = f"{veg_asset_folder}/{flux_folder}/{base_pattern}"
        ee_folder = f"wri_lgms/{veg_asset_folder}/{flux_folder}"

        if stack_vegetation_years:
            add_stacked_spec(
                key=f"vegetation_{flux}_{gas}_{years[0]}_{years[-1]}",
                dataset="vegetation",
                flux=flux,
                gas=gas,
                periods=years,
                pattern=pattern,
                gcs_dir=gcs_dir,
                ee_folder=ee_folder,
                version=f"v{veg_model_version_underscore}",
            )
        else:
            for year in years:
                add_spec(
                    key=f"vegetation_{flux}_{gas}_{year}",
                    dataset="vegetation",
                    flux=flux,
                    gas=gas,
                    period=year,
                    pattern=pattern,
                    gcs_dir=gcs_dir,
                    ee_folder=ee_folder,
                    version=f"v{veg_model_version_underscore}",
                )


# --------------------------------------------------------------------------------------------------
# Mineral soil
# Uses the final multi-year COG interval directly for both the GCS COG filename and the Earth Engine asset name (e.g. 2016_2020).
mineral_configs = {
    "emissions": cn.SOC_loss_min_soil_extent_pattern,
    "removals": cn.SOC_gain_min_soil_extent_pattern,
    "netflux": cn.SOC_net_min_soil_extent_pattern,
}

if "mineral_soil" in datasets and ("all_gases" in gases or "CO2_only" in gases):
    for period in mineral_soil_years:
        for flux, base_pattern in mineral_configs.items():
            if flux not in fluxes:
                continue

            add_spec(
                key=f"mineral_soil_{flux}_{period}",
                dataset="mineral_soil",
                flux=flux,
                gas="CO2_only",
                period=period,
                pattern=f"{base_pattern}_ha_yr",
                gcs_dir=f"{min_soil_asset_folder}/{base_pattern}",
                ee_folder=f"wri_lgms/{min_soil_asset_folder}/{flux}",
                version=f"v{cn.SOC_model_version_underscore}",
            )


# --------------------------------------------------------------------------------------------------
# Organic soil
# Organic soil only has emissions outputs, split into burned and drained components.
# When all years are selected, stack the two reporting intervals into one asset per component.
organic_configs = []

if "all_gases" in gases:
    organic_configs += [
        ("burned",  "all_gases", org_soil_burned_all_gases_pattern),
        ("drained", "all_gases", org_soil_drained_all_gases_pattern),
    ]

if "CO2_only" in gases:
    organic_configs += [
        ("burned",  "CO2_only", org_soil_burned_CO2_only_pattern),
        ("drained", "CO2_only", org_soil_drained_CO2_only_pattern),
    ]

# if "non_CO2_only" in gases:
#     organic_configs += [
#         ("burned",  "non_CO2_only", org_soil_burned_non_CO2_only_pattern),
#         ("drained", "non_CO2_only", org_soil_drained_non_CO2_only_pattern),
#     ]

stack_organic_soil_years = (
    years == all_vegetation_years
    or organic_soil_years == cn.organic_soil_year_intervals
)

if "organic_soil" in datasets and "emissions" in fluxes:
    for component, gas, pattern in organic_configs:

        gcs_dir = f"{org_soil_asset_folder}/{pattern}"
        ee_folder = f"wri_lgms/{org_soil_asset_folder}/emissions/{component}"

        if stack_organic_soil_years:
            add_stacked_spec(
                key=f"organic_soil_emissions_{component}_{gas}_{organic_soil_years[0]}_{organic_soil_years[-1]}",
                dataset="organic_soil",
                flux="emissions",
                gas=gas,
                component=component,
                periods=organic_soil_years,
                pattern=pattern,
                gcs_dir=gcs_dir,
                ee_folder=ee_folder,
                version=f"v{cn.organic_soil_model_version_underscore}",
            )

        else:
            for period in organic_soil_years:
                add_spec(
                    key=f"organic_soil_emissions_{component}_{gas}_{period}",
                    dataset="organic_soil",
                    flux="emissions",
                    gas=gas,
                    component=component,
                    period=period,
                    pattern=pattern,
                    gcs_dir=gcs_dir,
                    ee_folder=ee_folder,
                    version=f"v{cn.organic_soil_model_version_underscore}",
                )

print(f"Built {len(asset_specs)} Earth Engine registration specifications.")


Built 3 Earth Engine registration specifications.


In [5]:
# Print input COG(s) and Earth Engine asset paths
for key, spec in asset_specs.items():
    print(f"\n{key}")
    for period, gcs_uri in zip(spec["periods"], spec["gcs_uris"]):
        band_label = f"band {period}" if spec["stacked"] else "COG"
        print(f"  {band_label}: {gcs_uri}")
    print(f"  EE asset: {spec['asset_id']}")


vegetation_emissions_CO2_only_2016_2024
  band 2016: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2016.tif
  band 2017: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2017.tif
  band 2018: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2018.tif
  band 2019: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2019.tif
  band 2020: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2020.tif
  band 2021: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_p

In [6]:
# Verify that every backing COG exists in GCS
missing_cogs = []
found_count = 0
total_cogs = sum(len(spec["gcs_uris"]) for spec in asset_specs.values())

for key, spec in asset_specs.items():
    for gcs_uri in spec["gcs_uris"]:
        object_name = gcs_uri.replace(f"gs://{gcs_bucket}/", "", 1)
        exists = bucket.blob(object_name).exists()

        print(f"{key}: {gcs_uri} -> exists={exists}")

        if exists:
            found_count += 1
        else:
            missing_cogs.append(gcs_uri)

print(f"\nCOGs found: {found_count}/{total_cogs}")
if missing_cogs:
    print("\nMissing COGs:")
    for uri in missing_cogs:
        print(f"  {uri}")

vegetation_emissions_CO2_only_2016_2024: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2016.tif -> exists=True
vegetation_emissions_CO2_only_2016_2024: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2017.tif -> exists=True
vegetation_emissions_CO2_only_2016_2024: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2018.tif -> exists=True
vegetation_emissions_CO2_only_2016_2024: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2019.tif -> exists=True
vegetation_emissions_CO2_only_2016_2024: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__C

## Register the COGs in GEE

Each GeoTIFF is registered as its own COG-backed `ee.Image` asset. The source remains in Google Cloud Storage; Earth Engine references the external COG rather than ingesting a copy.

Information on image manifests and registering COGs as assets:
* https://developers.google.com/earth-engine/guides/image_manifest
* https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff

In [7]:
# Create an Earth Engine folder hierarchy if it does not already exist
def ensure_ee_folder(folder_path, ee_project):
    parts = folder_path.strip("/").split("/")
    current_path = f"projects/{ee_project}/assets"

    for part in parts:
        current_path = f"{current_path}/{part}"

        try:
            ee.data.getAsset(current_path)
            print(f"Exists:  {current_path}")
        except ee.EEException:
            ee.data.createAsset({"type": "FOLDER"}, current_path)
            print(f"Created: {current_path}")

# Specify year_ or period_ for veg vs soil data
def get_band_id(spec, period):
    prefix = "year" if spec["dataset"] == "vegetation" else "period"
    return f"{prefix}_{period}"

# Function to execute COG registration request
def register_cog_as_ee_asset(session, request, ee_project):
    url = f"https://earthengine.googleapis.com/v1alpha/projects/{ee_project}/image:importExternal"
    response = session.post(url=url, data=json.dumps(request))
    return json.loads(response.content)

# Make assets public
def make_ee_asset_public(asset_id):
    ee.data.setAssetAcl(
        asset_id,
        {
            "all_users_can_read": True
        }
    )
    print(f"PUBLIC: {asset_id}")

In [8]:
# Create every required Earth Engine folder once.
for folder in sorted({spec["asset_folder"] for spec in asset_specs.values()}):
    ensure_ee_folder(folder, ee_project)

Exists:  projects/landandcarbon/assets/wri_lgms
Exists:  projects/landandcarbon/assets/wri_lgms/soil
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/burned
Exists:  projects/landandcarbon/assets/wri_lgms
Exists:  projects/landandcarbon/assets/wri_lgms/soil
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions
Exists:  projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/drained
Exists:  projects/landandcarbon/assets/wri_lgms
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation
Exists:  projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5
Exists:  projects/landandcarbon/assets/w

In [9]:
# Build the Earth Engine image manifests.
requests = {}

for key, spec in asset_specs.items():
    properties = {
        "dataset": spec["dataset"],
        "flux": spec["flux"],
        "gas": spec["gas"],
        "period": str(spec["period"]),
        "version": spec["version"],
    }

    if spec["component"] is not None:
        properties["component"] = spec["component"]

    manifest = {
        "name": spec["asset_id"],
        "startTime": spec["start_time"],
        "endTime": spec["end_time"],
        "properties": properties,
    }

    if spec["stacked"]:
        manifest["tilesets"] = [
            {
                "id": get_band_id(spec, period),
                "sources": [{"uris": [gcs_uri]}],
            }
            for period, gcs_uri in zip(spec["periods"], spec["gcs_uris"])
        ]

        manifest["bands"] = [
            {
                "id": get_band_id(spec, period),
                "tilesetId": get_band_id(spec, period),
                "tilesetBandIndex": 0,
            }
            for period in spec["periods"]
        ]

    else:
        band_id = get_band_id(spec, spec["period"])

        manifest["tilesets"] = [
            {
                "id": band_id,
                "sources": [{"uris": [spec["gcs_uris"][0]]}]
            }
        ]

        manifest["bands"] = [
            {
                "id": band_id,
                "tilesetId": band_id,
                "tilesetBandIndex": 0,
            }
        ]

        if isinstance(spec["period"], int):
            properties["year"] = spec["period"]

    requests[key] = {"imageManifest": manifest}

print(f"Built {len(requests)} Earth Engine registration requests.")

# Preview stacked band names.
for key, request in requests.items():
    bands = request["imageManifest"].get("bands")
    if bands:
        print(f"{key} bands: {[band['id'] for band in bands]}")

Built 3 Earth Engine registration requests.
vegetation_emissions_CO2_only_2016_2024 bands: ['year_2016', 'year_2017', 'year_2018', 'year_2019', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'year_2024']
organic_soil_emissions_burned_CO2_only_2016_2020_2021_2024 bands: ['period_2016_2020', 'period_2021_2024']
organic_soil_emissions_drained_CO2_only_2016_2020_2021_2024 bands: ['period_2016_2020', 'period_2021_2024']


## Execute registration

This cell registers every configured COG. It skips an asset if that Earth Engine asset ID already exists, making the cell safer to rerun.

In [10]:
results = {}

for key, request in requests.items():
    asset_id = asset_specs[key]["asset_id"]

    try:
        ee.data.getAsset(asset_id)
        print(f"SKIP - already exists: {asset_id}")

        make_ee_asset_public(asset_id)  # Makes existing asset public if not already public

        results[key] = {"status": "already_exists", "asset_id": asset_id}
        continue

    except ee.EEException:
        pass

    if asset_specs[key]["stacked"]:
        print(f"REGISTER STACKED ASSET ({len(asset_specs[key]['gcs_uris'])} COGs):")
        for period, gcs_uri in zip(asset_specs[key]["periods"], asset_specs[key]["gcs_uris"]):
            print(f"  {period}: {gcs_uri}")
    else:
        print(f"REGISTER: {asset_specs[key]['gcs_uris'][0]}")

    print(f"      -> {asset_id}")

    results[key] = register_cog_as_ee_asset(session, request, ee_project)
    pprint(results[key])

    # Make newly registered asset public
    make_ee_asset_public(asset_id)

REGISTER STACKED ASSET (9 COGs):
  2016: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2016.tif
  2017: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2017.tif
  2018: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2018.tif
  2019: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2019.tif
  2020: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2020.tif
  2021: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2021.tif
 

In [11]:
folders = set()

for spec in asset_specs.values():
    parent = spec["asset_id"].rsplit("/", 1)[0]
    parts = parent.split("/")
    assets_index = parts.index("assets")

    for i in range(assets_index + 2, len(parts) + 1):
        folders.add("/".join(parts[:i]))

folders = sorted(folders, key=lambda x: (x.count("/"), x))

for folder in folders:
    info = ee.data.getAsset(folder)
    ee.data.setAssetAcl(info["name"], {"all_users_can_read": True})
    print(f"PUBLIC FOLDER: {info['name']}")

PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil/organic
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/emissions
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/burned
PUBLIC FOLDER: projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/drained


## Verify the registered assets


In [12]:
# Print the expected Earth Engine asset IDs and verify band names
for key, spec in asset_specs.items():
    print(f"\n{key}")
    for period, gcs_uri in zip(spec["periods"], spec["gcs_uris"]):
        band_label = f"band {period}" if spec["stacked"] else "COG"
        print(f"  {band_label}: {gcs_uri}")
    print(f"  EE asset: {spec['asset_id']}")

    try:
        info = ee.data.getAsset(spec["asset_id"])
        print(f"  EE type:  {info.get('type')}")
        print(f"  Bands:    {ee.Image(spec['asset_id']).bandNames().getInfo()}")
    except ee.EEException as exc:
        print(f"  NOT FOUND in Earth Engine: {exc}")


vegetation_emissions_CO2_only_2016_2024
  band 2016: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2016.tif
  band 2017: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2017.tif
  band 2018: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2018.tif
  band 2019: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2019.tif
  band 2020: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr_2020.tif
  band 2021: gs://wri-lcl-lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__CO2_only__MgCO2/gross_emissions__all_C_p

## Optional code to delete a registered asset:

In [102]:
# assets_to_delete = [
#      "projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/burned/burned_total_Mg_CO2e_ha_yr_2016_2024",
#      "projects/landandcarbon/assets/wri_lgms/soil/organic/v1_0_1/emissions/drained/drained_total_Mg_CO2e_ha_yr_2016_2024"
# ]
#
# for asset_id in assets_to_delete:
#     print(f"Deleting: {asset_id}")
#     ee.data.deleteAsset(asset_id)